<a href="https://colab.research.google.com/github/Ceciliah-Mumbi/cov_layers_extraction_automation/blob/main/GEE_Master_Extraction_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GEE Master Extraction Pipeline
**Combined from:** `01_Sentinel_2_MSI_1C` · `02_MOD13Q1` · `03_MCD64A1` · `05_CHIRPS` · `08_LS8_C2_T1_SR` · `14_OpenLandMap_CLM` · `16_SPL4SMGP`

| Layer ID | Name | Source | Scale |
|----------|------|--------|-------|
| RS_001 | NDVI - Sentinel-2 | COPERNICUS/S2_SR_HARMONIZED | 10m |
| RS_002 | NDVI - MODIS | MOD13Q1.061 | 250m |
| RS_003 | EVI - MODIS | MOD13Q1.061 | 250m |
| RS_004 | Burned Area | MCD64A1.061 | 500m |
| RS_012 | Annual Precipitation | CHIRPS Daily | 500m |
| RS_017 | NDVI - Landsat 8 | LC08/C02/T1_L2 | 30m |
| RS_018 | EVI - Landsat 8 | LC08/C02/T1_L2 | 30m |
| RS_019 | MSAVI - Landsat 8 | LC08/C02/T1_L2 | 30m |
| RS_047 | Monthly Precipitation | OpenLandMap CLM | 1000m |
| RS_049 | Top Soil Moisture | SMAP SPL4SMGP | 10000m |
| RS_050 | Root Soil Moisture | SMAP SPL4SMGP | 10000m |
| RS_051 | Top Soil Wetness | SMAP SPL4SMGP | 10000m |
| RS_052 | Root Soil Wetness | SMAP SPL4SMGP | 10000m |

---
### How to use
1. **Cell 1** — Install & authenticate
2. **Cell 2** — ⚠️ Edit configuration: year range, seasons, layers to run
3. **Cell 3** — Shared functions (always run)
4. **Cells 4–10** — Run each layer group independently
5. **Cell 11** — Check task status

### Updating layers each year
Only change `start_year` and `end_year` in Cell 2, then re-run the relevant layer cells.

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 1 — Install & Authenticate
# ════════════════════════════════════════════════════════════════
import ee

ee.Authenticate()
ee.Initialize(project = 'cmumbi')
print('✓ Earth Engine ready')

✓ Earth Engine ready


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 2 — ⚠️  CONFIGURATION  — edit this cell every update cycle
# ════════════════════════════════════════════════════════════════

# ── Area of interest ────────────────────────────────────────────
aoi      = ee.FeatureCollection("projects/cmumbi/assets/LLBN_buffered")
aoi_name = "LLBN"

# ── Export folder in Google Drive ───────────────────────────────
DRIVE_FOLDER = "GEE_exports"

# ── Date range ──────────────────────────────────────────────────
# TO UPDATE ANNUALLY: change start_year and end_year only
start_year = 2025   # <── change this each year
end_year   = 2025   # <── change this each year
year_list  = ee.List(list(range(start_year, end_year + 1)))

# ── Season parameters (months) ──────────────────────────────────
# eg. East Africa defaults Dry = Jul–Oct | Wet/Rain = Mar–May
rain_start = 3;  rain_end = 5
dry_start  = 7;  dry_end  = 10

# ── Reducer ─────────────────────────────────────────────────────
# Options: mean | median | min | max | stdDev | sum | product
img_col_reducer = "mean"
band_select     = ".*" + img_col_reducer

# ── Layers to run (set False to skip a layer group) ─────────────
RUN_LAYERS = {
    'RS_001_sentinel_ndvi':  True,
    'RS_002_003_modis_vi':   True,
    'RS_004_burn':           True,
    'RS_012_chirps':         True,
    'RS_017_018_019_landsat':True,
    'RS_047_openlandmap':    False,
    'RS_049_052_smap':       True,
}

print('✓ Configuration loaded')
print(f'  AOI        : {aoi_name}')
print(f'  Year range : {start_year}–{end_year}')
print(f'  Reducer    : {img_col_reducer}')
print(f'  Dry season : months {dry_start}–{dry_end}')
print(f'  Wet season : months {rain_start}–{rain_end}')
print(f'  Drive folder: {DRIVE_FOLDER}')

✓ Configuration loaded
  AOI        : LLBN
  Year range : 2025–2025
  Reducer    : mean
  Dry season : months 7–10
  Wet season : months 3–5
  Drive folder: GEE_exports


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 3 — Shared functions  (always run this cell)
# ════════════════════════════════════════════════════════════════

submitted_tasks = []  # tracks all submitted task names

# ── Shared multi-reducer (matches all original scripts) ─────────
reducer_list = (ee.Reducer.mean()
    .combine(reducer2=ee.Reducer.median(),  sharedInputs=True)
    .combine(reducer2=ee.Reducer.min(),     sharedInputs=True)
    .combine(reducer2=ee.Reducer.max(),     sharedInputs=True)
    .combine(reducer2=ee.Reducer.stdDev(),  sharedInputs=True)
    .combine(reducer2=ee.Reducer.sum(),     sharedInputs=True)
    .combine(reducer2=ee.Reducer.product(), sharedInputs=True))

# ── Export helper ────────────────────────────────────────────────
def export_to_drive(image, output_name, scale, region=None):
    """Export a single image to Google Drive."""
    region = region or aoi.geometry()
    task = ee.batch.Export.image.toDrive(
        image          = image,
        description    = "EXPORT IMAGE TO DRIVE",
        folder         = DRIVE_FOLDER,
        fileNamePrefix = output_name,
        scale          = scale,
        maxPixels      = 10e12,
        crs            = "EPSG:4326",
        region         = region
    )
    task.start()
    submitted_tasks.append(output_name)
    print(f'  STARTED: {output_name}')
    return task

# ── Season end date helper (avoids hardcoded day-30 bug) ────────
def season_end_date(year, month):
    """Returns the last day of a given month/year as ee.Date."""
    return ee.Date.fromYMD(year, month, 1).advance(1, 'month').advance(-1, 'day')

# ── Cloud masking — Sentinel-2 (QA60, matches original script) ──
def mask_s2_clouds(image):
    qa             = image.select('QA60')
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask= 1 << 11
    mask = (qa.bitwiseAnd(cloud_bit_mask).eq(0)
              .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0)))
    return image.updateMask(mask).divide(10000)

def add_ndvi_s2(image):
    return image.addBands(
        image.select('B.*').normalizedDifference(['B8','B4']).rename('NDVI'))

# ── Landsat-8 processing (matches original 08_LS8_C2_T1_SR) ────
def apply_scale_factors(image):
    return image.addBands(
        image.select('SR_B.').multiply(0.0000275).add(-0.2), None, True)

def rename_oli(image):
    return image.select(
        ['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7','QA_PIXEL'],
        ['Blue', 'Green','Red',  'NIR',  'SWIR1','SWIR2','QA_PIXEL'])

def fmask(image):
    qa   = image.select('QA_PIXEL')
    mask = (qa.bitwiseAnd(1 << 3).eq(0)
              .And(qa.bitwiseAnd(1 << 4).eq(0)))
    return image.updateMask(mask)

def add_ndvi_l8(image):
    return image.addBands(
        image.normalizedDifference(['NIR','Red']).rename('NDVI'))

def add_evi_l8(image):
    return image.addBands(image.expression(
        '2.5*((NIR-RED)/(NIR+6*RED-7.5*BLUE+1))',
        {'NIR':image.select('NIR'),'RED':image.select('Red'),'BLUE':image.select('Blue')}
    ).rename('EVI'))

def add_msavi_l8(image):
    return image.addBands(image.expression(
        '(2*NIR+1-sqrt(pow((2*NIR+1),2)-8*(NIR-RED)))/2',
        {'NIR':image.select('NIR'),'RED':image.select('Red')}
    ).rename('MSAVI'))

def prep_oli(image):
    orig  = image
    image = apply_scale_factors(image)
    image = rename_oli(image)
    image = fmask(image)
    image = add_ndvi_l8(image)
    image = add_evi_l8(image)
    image = add_msavi_l8(image)
    return ee.Image(image.copyProperties(orig, orig.propertyNames()))

print('✓ Shared functions ready')

✓ Shared functions ready


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 4 — RS_001: Sentinel-2 NDVI
# Source  : COPERNICUS/S2_SR_HARMONIZED
# Outputs : annual + dry + wet seasonal composites
# Scale   : 10m
# ════════════════════════════════════════════════════════════════
if not RUN_LAYERS['RS_001_sentinel_ndvi']:
    print('RS_001 skipped (set to False in config)')

else:
    print('RS_001 — Sentinel-2 NDVI')
    layer_name = 'RS_001'

    # -----------------------------
    # Sentinel-2 NDVI collection
    # -----------------------------
    sentinel2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                 .filterBounds(aoi)
                 .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
                 .map(mask_s2_clouds)
                 .map(add_ndvi_s2)
                 .select('NDVI'))

    # -----------------------------
    # ANNUAL NDVI
    # -----------------------------
    def s2_annual(year_date):
        start = ee.Date.fromYMD(year_date, 1, 1)
        end   = ee.Date.fromYMD(year_date, 12, 31)

        img = (sentinel2
               .filterDate(start, end)
               .mean()
               .clip(aoi)
               .set({
                   'name': start.format('YYYY_MM')
                           .cat('_to_')
                           .cat(end.format('YYYY_MM')),
                   'year': year_date
               }))

        return img

    annual_list = ee.ImageCollection(year_list.map(s2_annual))
    annual_list = annual_list.toList(annual_list.size())
    n_annual = annual_list.size().getInfo()

   # Annual section
for i in range(n_annual):
    img = ee.Image(annual_list.get(i))
    name_tag = img.get('name').getInfo()
    name = f"{layer_name}_{aoi_name}_NDVI_mean_{name_tag}"

    # Only export if image has bands
    n_bands = img.bandNames().size().getInfo()
    if n_bands == 0:
        print(f"  ⚠ Skipping {name} — no valid pixels (empty composite)")
        continue

    export_to_drive(img, name, scale=10)
    # -----------------------------
    # SEASONAL NDVI
    # -----------------------------
    def s2_seasonal(year_date, s_start, s_end, s_label):
        start = ee.Date.fromYMD(year_date, s_start, 1)
        end   = season_end_date(year_date, s_end)

        img = (sentinel2
               .filterDate(start, end)
               .mean()
               .clip(aoi)
               .set({
                   'season': s_label,
                   'year': year_date
               }))

        return img

    for s_label, s_start, s_end in [('dry', dry_start, dry_end),
                                     ('wet', rain_start, rain_end)]:

        # Modified the mapping to use a lambda function that explicitly takes one argument
        img_list = ee.ImageCollection(year_list.map(
            lambda y: s2_seasonal(y, s_start, s_end, s_label)
        ))
        img_list = img_list.toList(img_list.size())
        n_season = img_list.size().getInfo()

        # Seasonal section
for i in range(n_season):
    img = ee.Image(img_list.get(i))
    yr = img.get('year').getInfo()
    name = f"{layer_name}_{aoi_name}_NDVI_mean_{yr}_{s_label}"

    # Only export if image has bands
    n_bands = img.bandNames().size().getInfo()
    if n_bands == 0:
        print(f"  ⚠ Skipping {name} — no valid pixels (empty composite)")
        continue

    export_to_drive(img, name, scale=10)

    print('✓ RS_001 NDVI exports submitted successfully')
    #print(f'✓ RS_001 tasks submitted')

RS_001 — Sentinel-2 NDVI
  ⚠ Skipping RS_001_LLBN_NDVI_mean_2025_01_to_2025_12 — no valid pixels (empty composite)
  ⚠ Skipping RS_001_LLBN_NDVI_mean_2025_wet — no valid pixels (empty composite)


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 5 — RS_002 & RS_003: MODIS NDVI & EVI
# Source  : MODIS/061/MOD13Q1
# Outputs : annual + dry + wet seasonal composites
# Scale   : 250m
# ════════════════════════════════════════════════════════════════
if not RUN_LAYERS['RS_002_003_modis_vi']:
    print('RS_002/003 skipped')
else:
    print('RS_002/RS_003 — MODIS NDVI & EVI')
    modis = ee.ImageCollection('MODIS/061/MOD13Q1')

    def modis_annual(band_layer, layer_name):
        def _fn(year_date):
            start = ee.Date.fromYMD(year_date, 1, 1)
            end   = ee.Date.fromYMD(year_date, 12, 31)
            name  = start.format('YYYY_MM').cat('_to_').cat(end.format('YYYY_MM'))
            return (modis.filterDate(ee.DateRange(start, end))
                         .select(band_layer)
                         .reduce(reducer=reducer_list)
                         .multiply(0.0001)
                         .clip(aoi)
                         .set({'name': name}))
        return year_list.map(_fn)

    def modis_seasonal(band_layer):
        def _fn(year_date, s_start, s_end, s_label):
            start = ee.Date.fromYMD(year_date, s_start, 1)
            end   = season_end_date(year_date, s_end)
            return (modis.filterDate(ee.DateRange(start, end))
                         .select(band_layer)
                         .reduce(reducer=reducer_list)
                         .multiply(0.0001)
                         .clip(aoi)
                         .set({'season': s_label, 'year': year_date}))
        dry = year_list.map(lambda y: _fn(y, dry_start,  dry_end,  'dry'))
        wet = year_list.map(lambda y: _fn(y, rain_start, rain_end, 'wet'))
        return dry, wet

    for band_layer, layer_name in [('NDVI', 'RS_002'), ('EVI', 'RS_003')]:
        # Annual
        ann = modis_annual(band_layer, layer_name)
        for i in range(ee.List.length(ann).getInfo()):
            img  = ee.Image(ann.get(i)).select(band_select)
            name = f"{layer_name}_{aoi_name}_{img_col_reducer}_{ee.String(img.get('name')).getInfo()}"
            export_to_drive(img, name, scale=250)
        # Seasonal
        dry_list, wet_list = modis_seasonal(band_layer)
        for s_label, img_list in [('dry', dry_list), ('wet', wet_list)]:
            for i in range(ee.List.length(img_list).getInfo()):
                img  = ee.Image(img_list.get(i)).select(band_select)
                yr   = ee.String(img.get('year')).getInfo()
                name = f"{layer_name}_{aoi_name}_{img_col_reducer}_{yr}_{s_label}"
                export_to_drive(img, name, scale=250)

    print('✓ RS_002/003 tasks submitted')

RS_002/RS_003 — MODIS NDVI & EVI
  STARTED: RS_002_LLBN_mean_2025_01_to_2025_12
  STARTED: RS_002_LLBN_mean_2025_dry
  STARTED: RS_002_LLBN_mean_2025_wet
  STARTED: RS_003_LLBN_mean_2025_01_to_2025_12
  STARTED: RS_003_LLBN_mean_2025_dry
  STARTED: RS_003_LLBN_mean_2025_wet
✓ RS_002/003 tasks submitted


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 6 — RS_004: Burned Area
# Source  : MODIS/061/MCD64A1
# Outputs : annual max Julian day of burn
# Scale   : 500m  (original script used 30m — corrected to native 500m)
# ════════════════════════════════════════════════════════════════
if not RUN_LAYERS['RS_004_burn']:
    print('RS_004 skipped')
else:
    print('RS_004 — Burned Area (MCD64A1)')
    layer_name = 'RS_004'
    mcd64      = ee.ImageCollection('MODIS/061/MCD64A1')

    def burn_annual(year_date):
        start = ee.Date.fromYMD(year_date, 1, 1)
        end   = ee.Date.fromYMD(year_date, 12, 31)
        name  = start.format('YYYY_MM').cat('_to_').cat(end.format('YYYY_MM'))
        return (mcd64.filterDate(ee.DateRange(start, end))
                     .select('BurnDate')
                     .reduce(reducer=ee.Reducer.max())
                     .clip(aoi)
                     .set({'name': name}))

    annual_burn = year_list.map(burn_annual)
    for i in range(ee.List.length(annual_burn).getInfo()):
        img  = ee.Image(annual_burn.get(i))
        name = f"{layer_name}_{aoi_name}_{ee.String(img.get('name')).getInfo()}"
        export_to_drive(img, name, scale=500, region=aoi.geometry().bounds())

    print('✓ RS_004 tasks submitted')

RS_004 — Burned Area (MCD64A1)
  STARTED: RS_004_LLBN_2025_01_to_2025_12
✓ RS_004 tasks submitted


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 7 — RS_012: CHIRPS Annual Precipitation
# Source  : UCSB-CHG/CHIRPS/DAILY
# Outputs : annual sum per year
# Scale   : 500m (matches original)
# ════════════════════════════════════════════════════════════════
if not RUN_LAYERS['RS_012_chirps']:
    print('RS_012 skipped')
else:
    print('RS_012 — CHIRPS Annual Precipitation')
    layer_name = 'RS_012'
    chirps     = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
    years      = ee.List.sequence(start_year, end_year)

    def yearly_rain(focal_year):
        start = ee.Date.fromYMD(focal_year, 1, 1)
        end   = ee.Date.fromYMD(focal_year, 12, 31)
        return (chirps.filterDate(ee.DateRange(start, end))
                      .sum()
                      .clip(aoi))

    rain_years  = ee.ImageCollection.fromImages(years.map(yearly_rain).flatten())
    rain_bands  = rain_years.toBands()
    names_from  = rain_bands.bandNames().getInfo()
    names_to    = [str(y) for y in range(start_year, end_year + 1)]
    rain_bands  = rain_bands.select(names_from).rename(names_to)

    for yr_str in names_to:
        band = rain_bands.select(yr_str)
        name = f"{layer_name}_{aoi_name}_sum_{yr_str}"
        export_to_drive(band, name, scale=500, region=aoi.geometry().bounds())

    print('✓ RS_012 tasks submitted')

RS_012 — CHIRPS Annual Precipitation
  STARTED: RS_012_LLBN_sum_2025
✓ RS_012 tasks submitted


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 8 — RS_017 / RS_018 / RS_019: Landsat-8 NDVI, EVI, MSAVI
# Source  : LANDSAT/LC08/C02/T1_L2
# Outputs : annual + dry + wet seasonal composites per index
# Scale   : 30m
# ════════════════════════════════════════════════════════════════
if not RUN_LAYERS['RS_017_018_019_landsat']:
    print('RS_017/018/019 skipped')
else:
    print('RS_017/018/019 — Landsat-8 NDVI, EVI, MSAVI')

    layer_dict = {'NDVI': 'RS_017', 'EVI': 'RS_018', 'MSAVI': 'RS_019'}

    l8 = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
            .filterBounds(aoi)
            .map(prep_oli))  # applies scale, rename, fmask, NDVI, EVI, MSAVI

    def l8_annual(band_layer, year_date):
        start = ee.Date.fromYMD(year_date, 1, 1)
        end   = ee.Date.fromYMD(year_date, 12, 31)
        name  = start.format('YYYY_MM').cat('_to_').cat(end.format('YYYY_MM'))
        return (l8.filterDate(ee.DateRange(start, end))
                  .select(band_layer)
                  .reduce(reducer=reducer_list)
                  .clip(aoi)
                  .set({'name': name}))

    def l8_seasonal(band_layer, year_date, s_start, s_end, s_label):
        start = ee.Date.fromYMD(year_date, s_start, 1)
        end   = season_end_date(year_date, s_end)
        return (l8.filterDate(ee.DateRange(start, end))
                  .select(band_layer)
                  .reduce(reducer=reducer_list)
                  .clip(aoi)
                  .set({'season': s_label, 'year': year_date}))

    for band_layer, layer_name in layer_dict.items():
        # Annual
        ann = year_list.map(lambda y: l8_annual(band_layer, y))
        for i in range(ee.List.length(ann).getInfo()):
            img  = ee.Image(ann.get(i)).select(band_select)
            name = f"{layer_name}_{aoi_name}_{img_col_reducer}_{ee.String(img.get('name')).getInfo()}"
            export_to_drive(img, name, scale=30)
        # Seasonal
        for s_label, s_start, s_end in [('dry', dry_start, dry_end), ('wet', rain_start, rain_end)]:
            seas = year_list.map(lambda y: l8_seasonal(band_layer, y, s_start, s_end, s_label))
            for i in range(ee.List.length(seas).getInfo()):
                img  = ee.Image(seas.get(i)).select(band_select)
                yr   = ee.String(img.get('year')).getInfo()
                name = f"{layer_name}_{aoi_name}_{img_col_reducer}_{yr}_{s_label}"
                export_to_drive(img, name, scale=30)

    print('✓ RS_017/018/019 tasks submitted')

RS_017/018/019 — Landsat-8 NDVI, EVI, MSAVI
  STARTED: RS_017_LLBN_mean_2025_01_to_2025_12
  STARTED: RS_017_LLBN_mean_2025_dry
  STARTED: RS_017_LLBN_mean_2025_wet
  STARTED: RS_018_LLBN_mean_2025_01_to_2025_12
  STARTED: RS_018_LLBN_mean_2025_dry
  STARTED: RS_018_LLBN_mean_2025_wet
  STARTED: RS_019_LLBN_mean_2025_01_to_2025_12
  STARTED: RS_019_LLBN_mean_2025_dry
  STARTED: RS_019_LLBN_mean_2025_wet
✓ RS_017/018/019 tasks submitted


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 9 — RS_047: OpenLandMap Monthly Precipitation
# Source  : OpenLandMap/CLM/CLM_PRECIPITATION_SM2RAIN_M/v01
# Outputs : one image per monthly band (static product — no year loop)
# Scale   : 1000m
# Note    : This is a STATIC image (2007–2019 climatology), not yearly.
#           Re-running this layer every year is NOT needed unless the
#           product version changes.
# ════════════════════════════════════════════════════════════════
if not RUN_LAYERS['RS_047_openlandmap']:
    print('RS_047 skipped')
else:
    print('RS_047 — OpenLandMap Monthly Precipitation (static climatology)')
    layer_name = 'RS_047'

    rainfall  = ee.Image('OpenLandMap/CLM/CLM_PRECIPITATION_SM2RAIN_M/v01').clip(aoi)
    band_list = rainfall.bandNames().getInfo()
    print(f'  Bands found: {band_list}')

    for band in band_list:
        img  = rainfall.select(band)
        name = f"{layer_name}_{aoi_name}_{band}"
        export_to_drive(img, name, scale=1000)

    print('✓ RS_047 tasks submitted')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 10 — RS_049/050/051/052: SMAP Soil Moisture
# Source  : NASA/SMAP/SPL4SMGP/008
# Outputs : annual + dry + wet seasonal composites per band
# Scale   : 10000m
# ════════════════════════════════════════════════════════════════
if not RUN_LAYERS['RS_049_052_smap']:
    print('RS_049–052 skipped')
else:
    print('RS_049–052 — SMAP Soil Moisture & Wetness')

    smap = ee.ImageCollection('NASA/SMAP/SPL4SMGP/008').filterBounds(aoi)

    layer_dict_smap = {
        'sm_surface':          'RS_049',
        'sm_rootzone':         'RS_050',
        'sm_surface_wetness':  'RS_051',
        'sm_rootzone_wetness': 'RS_052',
    }

    def smap_seasonal(soil_band, year_date, s_start, s_end):
        start = ee.Date.fromYMD(year_date, s_start, 1)
        end   = season_end_date(year_date, s_end)
        return (smap.filterDate(ee.DateRange(start, end))
                    .select(soil_band)
                    .reduce(reducer=reducer_list)
                    .clip(aoi)
                    .set({'year': year_date, 'season_start': s_start}))

    def smap_annual(soil_band, year_date):
        start = ee.Date.fromYMD(year_date, 1, 1)
        end   = ee.Date.fromYMD(year_date, 12, 31)
        name  = start.format('YYYY_MM').cat('_to_').cat(end.format('YYYY_MM'))
        return (smap.filterDate(ee.DateRange(start, end))
                    .select(soil_band)
                    .reduce(reducer=reducer_list)
                    .clip(aoi)
                    .set({'name': name}))

    def export_smap(img_list, layer_name, season_label=None):
        for i in range(ee.List.length(img_list).getInfo()):
            img = ee.Image(img_list.get(i)).select(band_select)
            if season_label:
                yr   = ee.String(img.get('year')).getInfo()
                name = f"{layer_name}_{aoi_name}_{img_col_reducer}_{yr}_{season_label}"
            else:
                name = f"{layer_name}_{aoi_name}_{img_col_reducer}_{ee.String(img.get('name')).getInfo()}"
            export_to_drive(img, name, scale=10000)

    for soil_band, layer_name in layer_dict_smap.items():
        # Seasonal dry
        dry = year_list.map(lambda y: smap_seasonal(soil_band, y, dry_start, dry_end))
        export_smap(dry, layer_name, 'dry')
        # Seasonal wet
        wet = year_list.map(lambda y: smap_seasonal(soil_band, y, rain_start, rain_end))
        export_smap(wet, layer_name, 'wet')
        # Annual
        ann = year_list.map(lambda y: smap_annual(soil_band, y))
        export_smap(ann, layer_name)

    print('✓ RS_049–052 tasks submitted')


RS_049–052 — SMAP Soil Moisture & Wetness
  STARTED: RS_049_LLBN_mean_2025_dry
  STARTED: RS_049_LLBN_mean_2025_wet
  STARTED: RS_049_LLBN_mean_2025_01_to_2025_12
  STARTED: RS_050_LLBN_mean_2025_dry
  STARTED: RS_050_LLBN_mean_2025_wet
  STARTED: RS_050_LLBN_mean_2025_01_to_2025_12
  STARTED: RS_051_LLBN_mean_2025_dry
  STARTED: RS_051_LLBN_mean_2025_wet
  STARTED: RS_051_LLBN_mean_2025_01_to_2025_12
  STARTED: RS_052_LLBN_mean_2025_dry
  STARTED: RS_052_LLBN_mean_2025_wet
  STARTED: RS_052_LLBN_mean_2025_01_to_2025_12
✓ RS_049–052 tasks submitted


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 11 — Task status monitor
# Re-run this cell at any time to check progress
# ════════════════════════════════════════════════════════════════
tasks = ee.batch.Task.list()

counts = {'COMPLETED': 0, 'RUNNING': 0, 'READY': 0, 'FAILED': 0}
icons  = {'COMPLETED': '✓', 'RUNNING': '⟳', 'READY': '○', 'FAILED': '✗'}

# Show only tasks from this pipeline session
pipeline_tasks = [t for t in tasks
                  if t.status().get('description') == 'EXPORT IMAGE TO DRIVE']

print(f'\n{"═"*65}')
print(f'  Task Status Report  |  {len(submitted_tasks)} submitted this session')
print(f'{"═"*65}')

for task in pipeline_tasks[:50]:  # show latest 50
    s     = task.status()
    state = s.get('state', '?')
    icon  = icons.get(state, '?')
    counts[state] = counts.get(state, 0) + 1
    # Get file name from task id since description is generic
    task_id = s.get('id', '')[:8]
    print(f'  {icon} {state:<12} {task_id}')
    if state == 'FAILED':
        print(f'     ↳ {s.get("error_message", "check GEE Tasks tab")}')

total = sum(counts.values())
if total:
    pct = counts['COMPLETED'] / total * 100
    bar = '█' * int(pct / 2.5) + '░' * (40 - int(pct / 2.5))
    print(f'\n  [{bar}] {pct:.0f}%')
    print(f'  ✓ {counts["COMPLETED"]}  ⟳ {counts["RUNNING"]}  ○ {counts["READY"]}  ✗ {counts["FAILED"]}')

print(f'\n  Monitor live at: https://code.earthengine.google.com/tasks')


═════════════════════════════════════════════════════════════════
  Task Status Report  |  33 submitted this session
═════════════════════════════════════════════════════════════════
  ○ READY        C235NZKI
  ○ READY        JFDCYKFP
  ○ READY        XLNVBPLH
  ○ READY        4PTSIWKG
  ○ READY        FVIUOI7Y
  ○ READY        KR5L5EAA
  ○ READY        QLG5IZ2E
  ○ READY        EZ7XYZK2
  ○ READY        TLR5MJKN
  ○ READY        TA5JBIPR
  ○ READY        BSX27L5T
  ○ READY        P2LDXFKD
  ○ READY        5TXQWE3N
  ○ READY        M6BATG3Y
  ○ READY        GW22RF6Y
  ○ READY        AUG6ILYM
  ○ READY        ZPQUVL7F
  ○ READY        3JH5A4HJ
  ○ READY        PEA2FIQN
  ○ READY        HAT4PVPF
  ⟳ RUNNING      US7ZAOZ6
  ✓ COMPLETED    7E5D2YSF
  ✓ COMPLETED    6CHGRD6L
  ✓ COMPLETED    5UC2YQAH
  ✓ COMPLETED    DBCTLRS6
  ✓ COMPLETED    QMM3PG2Z
  ✓ COMPLETED    IXGDUDPO
  ✓ COMPLETED    THOHZDFE
  ✓ COMPLETED    JSZV77TD
  ✗ FAILED       CEP526MY
     ↳ Can't get band number 0. Imag